# Arithmetic with finite games

Combine games with Python operators, then evaluate the resulting game on any admissible coalition. When the operands support it, arithmetic also preserves access to Möbius coefficients and per-player contributions.

The examples use three players and a tiny coalition table, so every result can be checked by hand. See the [architecture guide](../docs/design/game-arithmetic.html) for dispatch, casting, and typing details.

In [1]:
import jax
import jax.numpy as jnp
from ndlike.jax_functions import jax_add
from ndlike.tuple.jax import JaxBroadcastTuple

from shapiq.game import AffineGame, DenseCoalitionGame, KnownAdditivity, MappedGame
from shapiq.index import ExactComputer, ShapleyValueIndex
from shapiq.players import FinitePlayers

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


## Compose and evaluate games

The table below represents a baseline of 1, singleton effects of 2, 3, and 4, and an extra 6 when players 0 and 1 both participate. Table entries follow little-endian coalition order. A boolean coalition uses the same canonical player order on its last axis.

Addition, subtraction, scaling, multiplication, and division are pointwise in the coalition. Building these expressions leaves coalition evaluation until a call to `eval` or `game(coalition)`.

In [2]:
players = FinitePlayers(3)
game = DenseCoalitionGame(jnp.array([1.0, 3.0, 4.0, 12.0, 5.0, 7.0, 8.0, 16.0]), players)
pair = jnp.array([True, True, False])

adjusted = 2 * (game + 1) - 3
product = game * (game + 1)
quotient = product / (game + 1)

assert jnp.allclose(adjusted(pair), 2 * (game(pair) + 1) - 3)
assert jnp.allclose(jax_add(game, 1)(pair), (game + 1)(pair))
assert jnp.allclose(product(pair), 156)
assert jnp.allclose(quotient(pair), game(pair))
print("Adjusted, product, quotient", adjusted(pair), product(pair), quotient(pair))

Adjusted, product, quotient 23.0 156.0 12.0


## Build wrappers directly

`AffineGame` accepts a `JaxBroadcastTuple` of games, one weight per game, and a bias. `MappedGame` receives a callable whose input is a tuple of broadcast-aligned coalition values. This map clips the blended score independently at each batch position.

In [3]:
blend = AffineGame(
    games=JaxBroadcastTuple((game, game + 4)),
    weights=JaxBroadcastTuple((jnp.asarray(0.25), jnp.asarray(0.75))),
    bias=jnp.asarray(-1.0),
)
clipped = MappedGame(JaxBroadcastTuple((blend,)), lambda values: jnp.minimum(values[0], 10.0))

assert jnp.allclose(blend(pair), game(pair) + 2)
assert jnp.allclose(clipped(pair), 10)
print("Blend and clipped blend", blend(pair), clipped(pair))

Blend and clipped blend 14.0 10.0


## Keep contributions through affine arithmetic

An exact Shapley explanation is itself an additive game. Scale it and add an offset to get another additive game. The offset changes `empty_value()` and every coalition evaluation. `contributions()` reads just the first-order Möbius coefficients in player order.

The arithmetic result preserves the known additivity bound (`1`) without recomputing coefficients. Use `isinstance(..., KnownAdditivity)` to check whether `additivity()` is O(1).

In [4]:
explanation = ExactComputer.from_game(game)(ShapleyValueIndex())
scaled = 2 * explanation + 3

assert isinstance(scaled, KnownAdditivity)
assert scaled.additivity() == 1
assert jnp.allclose(explanation.contributions(), jnp.array([5.0, 6.0, 4.0]))
assert jnp.allclose(scaled.contributions(), jnp.array([10.0, 12.0, 8.0]))
assert jnp.allclose(scaled.coefficients(order=1), scaled.contributions())
assert jnp.allclose(scaled.empty_value(), 3)
assert jnp.allclose(scaled.full_value(), 33)
print("Empty value", scaled.empty_value())
print("Contributions", scaled.contributions())
print("Full value", scaled.full_value())

Empty value 3.0
Contributions [10.000002 12.000003  8.000002]
Full value 33.000008


## Read higher-order coefficients

The pointwise product of two additive games has a Möbius order bound of at most two. Query all pairs with `order=2`, or pass a batch of explicit interactions. The explicit query below asks for the empty interaction, a singleton, a pair, and the triple, whose coefficient is zero.

In [5]:
quadratic = explanation * (explanation + 1)
assert isinstance(quadratic, KnownAdditivity)
assert quadratic.additivity() == 2

queries = jnp.stack(
    (players.empty_coalition(), jnp.array([True, False, False]), pair, players.full_coalition())
)
pairs = quadratic.coefficients(order=2)
selected = quadratic.coefficients(queries)

assert jnp.allclose(pairs, jnp.array([60.0, 40.0, 48.0]))
assert jnp.allclose(selected, jnp.array([0.0, 30.0, 60.0, 0.0]))
print("Pairs (0, 1), (0, 2), (1, 2)", pairs)
print("Empty, singleton, pair, triple", selected)

Pairs (0, 1), (0, 2), (1, 2) [60.000015 40.000015 48.000008]
Empty, singleton, pair, triple [ 0.       30.000011 60.000015  0.      ]


## Broadcast, reshape, and compile

A batch of two offsets creates two games. Give the three coalitions a singleton batch axis so each coalition is evaluated against each game. The result has shape `(3, 2)` with coalitions down the rows and games across the columns. Indexing selects a game and reshaping changes only the game batch axes.

Construct the expression outside JIT so compatibility checks can inspect player IDs when needed. Then pass the constructed game and coalitions as arguments to the compiled function.

In [6]:
batched = game + jnp.array([0.0, 10.0])
coalitions = jnp.stack(
    (players.empty_coalition(), jnp.array([True, False, False]), players.full_coalition())
)[:, None, :]


@jax.jit
def evaluate(game_batch, selections):
    return game_batch.eval(selections)


values = evaluate(batched, coalitions)
assert batched.shape == (2,)
assert batched[1].shape == ()
assert batched.reshape(2, 1).shape == (2, 1)
assert jnp.allclose(values, jnp.array([[1.0, 11.0], [3.0, 13.0], [16.0, 26.0]]))
print(values)

[[ 1. 11.]
 [ 3. 13.]
 [16. 26.]]


## Check player compatibility

Games can share a coalition when they agree on what each player position means. Compatibility compares ordered IDs after broadcasting, together with the latent player space and the identity of the optional shared catalog. A game on IDs `[0, 1, 3]` therefore cannot be added to a game on IDs `[0, 1, 2]`, even though both accept three-bit coalitions.

In [7]:
left_players = FinitePlayers(jnp.array([0, 1, 2]), latent_n_players=4)
right_players = FinitePlayers(jnp.array([0, 1, 3]), latent_n_players=4)
assert not left_players.is_compatible(right_players)

left = DenseCoalitionGame(game.values, left_players)
right = DenseCoalitionGame(game.values, right_players)
try:
    left + right
except ValueError as error:
    print(error)
else:
    error_message = "Expected the player sets to be incompatible."
    raise AssertionError(error_message)

ArithmeticGame requires compatible player sets.
